# Phase 2 — Pre-release hit prediction: feature engineering

**Phase 1 conclusion (closed):** the parent paper's R² = 1.000 was an arithmetic artifact
(`Global_Sales = NA + EU + JP + Other`). A corrected, leakage-free replication reached
R² ≈ 0.44 (CV) / 0.68 (Validation) — but leaned heavily on `User_Count`, which is itself a
*post-outcome* variable: user reviews accumulate after release, roughly in proportion to how
many people bought the game. That's a second, subtler leak — temporal rather than arithmetic —
and it means the Phase 1 model cannot forecast anything, since `User_Count` does not exist for
an unreleased title.

**Phase 2 reframes the question honestly:**

> Can we predict whether a game will be a commercial hit (≥1.0M units) using *only*
> information that exists before it ships?

This notebook builds and validates the feature set. **No models are fit here** — the point is
to confirm the engineered features are (a) temporally honest and (b) actually carry signal,
before any modeling is worth doing.

### The leakage discipline, one level deeper
Every historical feature below is computed on an **expanding window**: for a title released in
year *Y*, only titles released in years **strictly < Y** contribute. Computing "Final Fantasy
has 57 entries" from the full dataset and feeding it to a 2005 release would use 2015
information — the same class of mistake as the parent paper's, one level further in.

In [1]:
import os
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = os.path.join("..", "paper_replication", "video_game_sales_and_ratings.csv")
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

HIT_THRESHOLD = 1.0  # millions of units — the industry "million-seller" milestone
pd.set_option("display.width", 140)

## 1. Load and clean

Only announcement-time columns are required here, so rows missing `Critic_Score` are **kept**
— that alone takes the usable sample from 6,825 (Phase 1) back up to ~16,400, and removes the
"only well-reviewed titles" sample bias noted in the Phase 1 limitations.

In [2]:
raw = pd.read_csv(DATA_PATH)
df = raw.dropna(subset=["Name", "Genre", "Platform", "Publisher", "Year_of_Release"]).copy()
df["Year_of_Release"] = df["Year_of_Release"].astype(int)
df["Critic_Score"] = pd.to_numeric(df["Critic_Score"], errors="coerce")

df["is_hit"] = (df["Global_Sales"] >= HIT_THRESHOLD).astype(int)

print(f"Raw rows: {len(raw)}")
print(f"Usable rows (announcement-time columns only): {len(df)}")
print(f"  ... vs 6,825 in Phase 1, which required review data")
print(f"Unique titles: {df['Name'].nunique()}")
print()
print(f"Target: Global_Sales >= {HIT_THRESHOLD}M  ->  {df['is_hit'].mean():.1%} positive ({df['is_hit'].sum()} rows)")

Raw rows: 16719
Usable rows (announcement-time columns only): 16416
  ... vs 6,825 in Phase 1, which required review data
Unique titles: 11397

Target: Global_Sales >= 1.0M  ->  12.5% positive (2052 rows)


## 2. Franchise detection from `Name`

Stem each title by cutting at the first separator (`:`, `-`, `(`) and keeping the leading
words. This is a heuristic, not a curated franchise database — it groups *Final Fantasy VII*
and *Final Fantasy X* correctly, but also produces some ugly stems (Yu-Gi-Oh truncating at the
hyphen). Grouping accuracy matters here; the label text does not.

In [3]:
STOP_STEMS = {"the", "a", "an"}


def franchise_stem(name: str) -> str:
    n = str(name).lower().strip()
    n = re.split(r"[:\-\u2013(]", n)[0]
    n = re.sub(r"[^a-z0-9 ]", "", n)
    words = [w for w in n.split() if w]
    if not words:
        return ""
    # keep 3 leading words when the first is a stopword ("the legend of zelda"), else 2
    take = 3 if words[0] in STOP_STEMS else 2
    return " ".join(words[:take])


df["franchise"] = df["Name"].map(franchise_stem)

# Title-level view: one row per game. Multi-platform releases must not be counted repeatedly
# when measuring franchise history, and a title's true size is its sales summed over platforms.
titles = (
    df.groupby("Name")
    .agg(
        franchise=("franchise", "first"),
        Publisher=("Publisher", "first"),
        Genre=("Genre", "first"),
        year=("Year_of_Release", "min"),
        title_sales=("Global_Sales", "sum"),
    )
    .reset_index()
)

fr_counts = titles["franchise"].value_counts()
print(f"Distinct franchise stems: {len(fr_counts)}")
print(f"Stems with >=2 titles:    {(fr_counts >= 2).sum()}")
print(f"Share of titles in a multi-title franchise: {titles['franchise'].map(fr_counts).ge(2).mean():.1%}")
print()
fr_counts.head(12)

Distinct franchise stems: 6351
Stems with >=2 titles:    1556
Share of titles in a multi-title franchise: 57.9%



franchise
final fantasy       57
dragon ball         53
mega man            52
jikkyou powerful    52
star wars           46
yu                  46
super robot         39
the legend of       39
imagine             36
tom clancys         35
mobile suit         34
dynasty warriors    32
Name: count, dtype: int64

## 3. Expanding-window history features

For each title, aggregate over **strictly earlier years only**. Implemented as a cumulative
aggregate per year followed by an **as-of (backward) join queried at `year - 1`**, so:

- same-year releases never inform each other, and
- a franchise dormant since 2003 still contributes its history to a 2009 release.

(An earlier version joined on exact `(franchise, year)` matches, which silently dropped history
for ~4% of rows — later ports of titles whose franchise had no debut that year. The as-of join
fixes that.)

In [4]:
def cumulative_history(titles_df: pd.DataFrame, key: str) -> pd.DataFrame:
    """Per (key, year): cumulative count / sum / max of title sales THROUGH that year."""
    per_year = (
        titles_df.groupby([key, "year"])
        .agg(n=("title_sales", "size"), s=("title_sales", "sum"), mx=("title_sales", "max"))
        .reset_index()
        .sort_values([key, "year"])
    )
    g = per_year.groupby(key)
    per_year["cum_n"] = g["n"].cumsum()
    per_year["cum_s"] = g["s"].cumsum()
    per_year["cum_mx"] = g["mx"].cummax()
    return per_year[[key, "year", "cum_n", "cum_s", "cum_mx"]]


def attach_prior_history(rows: pd.DataFrame, hist: pd.DataFrame, key: str, prefix: str) -> pd.DataFrame:
    """As-of join: for a title released in year Y, carry forward the cumulative history from
    the most recent year STRICTLY BEFORE Y. Querying at Y-1 with a backward as-of match means
    a franchise that was dormant for years still contributes its history, while same-year
    releases never inform each other."""
    left = rows[[key, "Year_of_Release"]].copy()
    left["_query_year"] = left["Year_of_Release"] - 1
    left = left.reset_index().sort_values("_query_year")

    merged = pd.merge_asof(
        left,
        hist.sort_values("year"),
        left_on="_query_year",
        right_on="year",
        by=key,
        direction="backward",
    ).set_index("index").sort_index()

    out = pd.DataFrame(index=rows.index)
    out[f"{prefix}_count"] = merged["cum_n"].fillna(0)
    out[f"{prefix}_mean_sales"] = np.where(merged["cum_n"] > 0, merged["cum_s"] / merged["cum_n"], np.nan)
    out[f"{prefix}_max_sales"] = merged["cum_mx"]
    return out


franchise_hist = cumulative_history(titles, "franchise")
publisher_hist = cumulative_history(titles, "Publisher")

df = pd.concat([
    df,
    attach_prior_history(df, franchise_hist, "franchise", "prior_franchise"),
    attach_prior_history(df, publisher_hist, "Publisher", "prior_publisher"),
], axis=1)

df["is_sequel"] = (df["prior_franchise_count"] > 0).astype(int)

print("Prior-history feature coverage (non-null share):")
for c in ["prior_franchise_count", "prior_franchise_mean_sales", "prior_publisher_count", "prior_publisher_mean_sales"]:
    print(f"  {c:<32} {df[c].notna().mean():.1%}")
print()
print(f"Rows flagged as sequels / franchise entries: {df['is_sequel'].mean():.1%}")

Prior-history feature coverage (non-null share):
  prior_franchise_count            100.0%
  prior_franchise_mean_sales       47.4%
  prior_publisher_count            100.0%
  prior_publisher_mean_sales       94.5%

Rows flagged as sequels / franchise entries: 47.4%


### Leakage spot-check

The expanding window is the whole methodological contribution here, so verify it directly:
trace one franchise chronologically and confirm `prior_franchise_count` only ever reflects
*earlier* entries, and that the first entry has no history at all.

In [5]:
check = (
    df[df["franchise"] == "final fantasy"]
    .drop_duplicates("Name")
    .sort_values("Year_of_Release")
    [["Name", "Year_of_Release", "prior_franchise_count", "prior_franchise_mean_sales", "prior_franchise_max_sales"]]
)
check.head(12)

,Name,Year_of_Release,prior_franchise_count,prior_franchise_mean_sales,prior_franchise_max_sales
2461,Final Fantasy,1987,0.0,NaN,NaN
985,Final Fantasy II,1991,3.0,3.713333,7.01
2711,Final Fantasy Legend III,1991,3.0,3.713333,7.01
5014,Final Fantasy Adventure,1991,3.0,3.713333,7.01
632,Final Fantasy V,1992,5.0,2.456000,7.01
388,Final Fantasy III,1994,6.0,2.483333,7.01
10291,Final Fantasy I & II,1994,6.0,2.483333,7.01
3206,Final Fantasy VII International,1997,7.0,2.144286,7.01
633,Final Fantasy Tactics,1997,7.0,2.144286,7.01
65,Final Fantasy VII,1997,7.0,2.144286,7.01


## 4. Remaining announcement-time features

- **`simultaneous_platforms`** — how many platforms the title launches on *in the same year*.
  Knowable at announcement, and a proxy for publisher investment. (Counting a title's platforms
  across all years would leak: a 2012 PC port is not knowable at a 2010 console launch.)
- **`platform_age`** — years since that platform's first appearance, a proxy for install-base
  maturity.
- **`genre_density_prior_year`** — how crowded the genre was the year before, as a competition
  measure. Prior-year rather than same-year, to stay strictly announcement-safe.

In [6]:
sim = df.groupby(["Name", "Year_of_Release"]).size().rename("simultaneous_platforms").reset_index()
df = df.merge(sim, on=["Name", "Year_of_Release"], how="left")

platform_debut = df.groupby("Platform")["Year_of_Release"].min().rename("platform_debut")
df = df.merge(platform_debut, on="Platform", how="left")
df["platform_age"] = df["Year_of_Release"] - df["platform_debut"]

genre_year = titles.groupby(["Genre", "year"]).size().rename("n_titles").reset_index()
genre_year["Year_of_Release"] = genre_year["year"] + 1  # prior-year density
df = df.merge(
    genre_year[["Genre", "Year_of_Release", "n_titles"]].rename(columns={"n_titles": "genre_density_prior_year"}),
    on=["Genre", "Year_of_Release"], how="left",
)

print(df[["simultaneous_platforms", "platform_age", "genre_density_prior_year"]].describe().round(2))

       simultaneous_platforms  platform_age  genre_density_prior_year
count                16416.00      16416.00                  16366.00
mean                     1.93          7.48                     64.79
std                      1.48          8.03                     41.09
min                      1.00          0.00                      1.00
25%                      1.00          2.00                     30.00
50%                      1.00          4.00                     56.00
75%                      2.00          7.00                     98.00
max                      9.00         35.00                    172.00


## 5. Do these features actually carry signal?

The whole point of validating before modeling: if hit rate doesn't move across these features,
there is nothing for a classifier to learn and the direction isn't worth pursuing.

In [7]:
base_rate = df["is_hit"].mean()
print(f"Base hit rate: {base_rate:.1%}\n")

print("Hit rate by franchise status:")
print(df.groupby("is_sequel")["is_hit"].agg(["mean", "size"]).rename(index={0: "standalone", 1: "franchise entry"}).round(3))
print()

print("Hit rate by simultaneous platform count:")
print(df.groupby(df["simultaneous_platforms"].clip(upper=4))["is_hit"].agg(["mean", "size"]).round(3))
print()

for col in ["prior_franchise_mean_sales", "prior_publisher_mean_sales"]:
    sub = df[df[col].notna()].copy()
    sub["quartile"] = pd.qcut(sub[col], 4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])
    print(f"Hit rate by {col} quartile:")
    print(sub.groupby("quartile", observed=True)["is_hit"].agg(["mean", "size"]).round(3))
    print()

Base hit rate: 12.5%

Hit rate by franchise status:
                  mean  size
is_sequel                   
standalone       0.108  8631
franchise entry  0.144  7785

Hit rate by simultaneous platform count:
                         mean  size
simultaneous_platforms             
1                       0.117  9832
2                       0.101  2546
3                       0.138  1782
4                       0.176  2256

Hit rate by prior_franchise_mean_sales quartile:
               mean  size
quartile                 
Q1 (lowest)   0.017  1947
Q2            0.043  1950
Q3            0.155  1946
Q4 (highest)  0.360  1942

Hit rate by prior_publisher_mean_sales quartile:
               mean  size
quartile                 
Q1 (lowest)   0.017  3924
Q2            0.076  3836
Q3            0.151  3897
Q4 (highest)  0.271  3853



In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

d1 = df.groupby("is_sequel")["is_hit"].mean()
axes[0].bar(["Standalone", "Franchise entry"], d1.values, color=["#8C8C8C", "#55A868"])
axes[0].set_title("Hit rate: franchise vs standalone")

d2 = df.groupby(df["simultaneous_platforms"].clip(upper=4))["is_hit"].mean()
axes[1].bar(d2.index.astype(str), d2.values, color="#4C72B0")
axes[1].set_title("Hit rate by simultaneous platforms")
axes[1].set_xlabel("platforms at launch (4 = 4+)")

sub = df[df["prior_franchise_mean_sales"].notna()].copy()
sub["q"] = pd.qcut(sub["prior_franchise_mean_sales"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
d3 = sub.groupby("q", observed=True)["is_hit"].mean()
axes[2].bar(d3.index.astype(str), d3.values, color="#C44E52")
axes[2].set_title("Hit rate by prior franchise sales")

for ax in axes:
    ax.axhline(base_rate, ls="--", c="k", lw=1, label=f"base rate {base_rate:.1%}")
    ax.set_ylabel("P(hit)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "phase2_feature_signal.png"), dpi=150)
plt.close()
print("Saved outputs/phase2_feature_signal.png")

Saved outputs/phase2_feature_signal.png


## 6. A third leakage vector, flagged for the modeling stage

This dataset is **one row per game-platform release**, so a multi-platform title appears
several times with nearly identical features. A random train/test split would put *Call of
Duty* on PS3 in training and the same game on X360 in test — the model would effectively be
scored on games it has already seen.

The engineered `Name` column is therefore kept in the saved file as a **grouping key**: the
modeling notebook must split with `GroupKFold`/`GroupShuffleSplit` on `Name`, so every platform
variant of a title lands in the same fold. Noted here rather than discovered later.

In [9]:
FEATURE_COLUMNS = [
    # announcement-time categoricals
    "Genre", "Platform", "Publisher", "Rating", "Year_of_Release",
    # engineered, expanding-window
    "is_sequel", "prior_franchise_count", "prior_franchise_mean_sales", "prior_franchise_max_sales",
    "prior_publisher_count", "prior_publisher_mean_sales", "prior_publisher_max_sales",
    "simultaneous_platforms", "platform_age", "genre_density_prior_year",
    # launch-window (Tier 2 experiment only)
    "Critic_Score",
]
KEEP = ["Name", "franchise"] + FEATURE_COLUMNS + ["Global_Sales", "is_hit"]

phase2 = df[KEEP].copy()
phase2.to_csv("phase2_features.csv", index=False)

print(f"Saved phase2_features.csv  ({len(phase2)} rows, {len(FEATURE_COLUMNS)} features)")
print(f"  Tier 1 (announcement-only) rows: {len(phase2)}")
print(f"  Tier 2 (requires Critic_Score):  {phase2['Critic_Score'].notna().sum()}")
phase2.head()

Saved phase2_features.csv  (16416 rows, 16 features)
  Tier 1 (announcement-only) rows: 16416
  Tier 2 (requires Critic_Score):  7982


,Name,franchise,Genre,Platform,Publisher,Rating,Year_of_Release,is_sequel,prior_franchise_count,prior_franchise_mean_sales,prior_franchise_max_sales,prior_publisher_count,prior_publisher_mean_sales,prior_publisher_max_sales,simultaneous_platforms,platform_age,genre_density_prior_year,Critic_Score,Global_Sales,is_hit
0,Wii Sports,wii sports,Sports,Wii,Nintendo,E,2006,0,0.0,NaN,NaN,345.0,2.790261,45.31,1,0,69.0,76.0,82.53,1
1,Super Mario Bros.,super mario,Platform,NES,Nintendo,NaN,1985,0,0.0,NaN,NaN,16.0,4.488125,28.31,1,2,1.0,NaN,40.24,1
2,Mario Kart Wii,mario kart,Racing,Wii,Nintendo,E,2008,1,4.0,11.375,23.21,439.0,2.893280,82.53,1,2,43.0,82.0,35.52,1
3,Wii Sports Resort,wii sports,Sports,Wii,Nintendo,E,2009,1,1.0,82.530,82.53,470.0,2.893936,82.53,1,3,109.0,80.0,32.77,1
4,Pokemon Red/Pokemon Blue,pokemon redpokemon,Role-Playing,GB,Nintendo,NaN,1996,0,0.0,NaN,NaN,96.0,4.336458,45.31,1,8,24.0,NaN,31.37,1


## Next step

Modeling notebook: Logistic Regression + Random Forest classifiers on the Tier 1 feature set,
with `GroupKFold` on `Name`, reporting precision / recall / F1 / ROC-AUC against the 12.5%
base rate — then the Tier 1 vs Tier 1+Critic_Score comparison, and a chronological
train/validate split as a robustness check.